In [20]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [21]:
from pathlib import Path
from typing import Literal
from brain_image.data import load_all_eeg_data

split: Literal["train", "test"] = "test"
subs = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

sub_paths = [
    Path(
        f"data/things-eeg2/eeg/sub-{sub:02}/preprocessed_eeg_{'training' if split == 'train' else 'test'}.npy"
    )
    for sub in subs
]

eeg_data, times, channel_names = load_all_eeg_data(eeg_paths=sub_paths)

print(eeg_data.shape)
print(times.shape)
print(channel_names)

torch.Size([10, 200, 17, 100])
torch.Size([100])
['Pz', 'P3', 'P7', 'O1', 'Oz', 'O2', 'P4', 'P8', 'P1', 'P5', 'PO7', 'PO3', 'POz', 'PO4', 'PO8', 'P6', 'P2']


In [22]:
from brain_image.data import get_image_paths

img_paths = get_image_paths(
    image_dir=Path("data/things-eeg2/imgs"),
    split=split,
)

print(len(img_paths))

200


In [23]:
merged_data = []

for sub in subs:
    for i in range(len(img_paths)):
        img_path = img_paths[i]
        eeg = eeg_data[sub-1, i]

        joined_object = {
            "img_path": str(img_path),
            "eeg": eeg,
        }

        merged_data.append(joined_object)

print(len(merged_data))

2000


In [24]:
from pathlib import Path
import json

import torch


dst_path = Path(f"data/things-eeg2/prepared/{split}.pt")
dst_path.parent.mkdir(parents=True, exist_ok=True)

torch.save(merged_data, dst_path)

In [25]:
loaded_data = torch.load(Path(f"data/things-eeg2/prepared/{split}.pt"))

print(len(loaded_data))
print(loaded_data[0])

2000
{'img_path': 'data/things-eeg2/imgs/test_images/00001_aircraft_carrier/aircraft_carrier_06s.jpg', 'eeg': tensor([[ 9.7007e-02,  1.2778e-01,  1.0261e-01,  ...,  2.8108e-01,
          3.5363e-01,  3.4408e-01],
        [-5.6824e-02,  2.2840e-02, -2.3371e-02,  ..., -3.5719e-02,
         -2.6789e-02, -4.4720e-02],
        [-2.1760e-02,  5.7279e-03, -3.6088e-02,  ..., -2.7456e-01,
         -3.5171e-01, -3.5298e-01],
        ...,
        [-5.9338e-02, -1.0865e-01, -1.8249e-01,  ..., -3.0808e-01,
         -3.0247e-01, -2.8589e-01],
        [-1.1831e-02, -5.1971e-02, -1.3633e-01,  ..., -1.9957e-01,
         -1.2550e-01, -9.4855e-02],
        [ 5.3489e-02,  7.2398e-02, -1.6038e-04,  ...,  5.7487e-02,
          1.1296e-01,  1.4642e-01]])}
